In [5]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Feature Engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Train Test Split
from sklearn.model_selection import train_test_split

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [22]:
df = pd.read_csv("Amazon_Reviews.csv", engine='python', encoding='latin-1')

print(df.columns)  # confirm again

Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='object')


In [23]:
df = df[['Review Text', 'Rating']]
df.columns = ['review', 'rating']

print(df.head())

                                              review                  rating
0  I registered on the website, tried to order a ...  Rated 1 out of 5 stars
1  Had multiple orders one turned up and driver h...  Rated 1 out of 5 stars
2  I informed these reprobates that I WOULD NOT B...  Rated 1 out of 5 stars
3  I have bought from Amazon before and no proble...  Rated 1 out of 5 stars
4  If I could give a lower rate I would! I cancel...  Rated 1 out of 5 stars


In [24]:
df['rating'] = df['rating'].astype(str)
df['rating'] = df['rating'].str.extract('(\d)')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# Drop null
df.dropna(inplace=True)

# Convert to sentiment
def convert_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating == 3:
        return "neutral"
    else:
        return "negative"

df['sentiment'] = df['rating'].apply(convert_sentiment)

print(df.head())
print(df['sentiment'].value_counts())

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_15337/564325971.py:2: SyntaxWarning: invalid escape sequence '\d'
  df['rating'] = df['rating'].str.extract('(\d)')


                                              review  rating sentiment
0  I registered on the website, tried to order a ...     1.0  negative
1  Had multiple orders one turned up and driver h...     1.0  negative
2  I informed these reprobates that I WOULD NOT B...     1.0  negative
3  I have bought from Amazon before and no proble...     1.0  negative
4  If I could give a lower rate I would! I cancel...     1.0  negative
sentiment
negative    14350
positive     5820
neutral       885
Name: count, dtype: int64


In [25]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation & numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [27]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [28]:
df['cleaned_text'] = df['review'].apply(preprocess_text)

print(df[['review', 'cleaned_text']].head())

                                              review  \
0  I registered on the website, tried to order a ...   
1  Had multiple orders one turned up and driver h...   
2  I informed these reprobates that I WOULD NOT B...   
3  I have bought from Amazon before and no proble...   
4  If I could give a lower rate I would! I cancel...   

                                        cleaned_text  
0  registered website tried order laptop entered ...  
1  multiple order one turned driver phone door nu...  
2  informed reprobate would going visit sick rela...  
3  bought amazon problem happy service price amaz...  
4  could give lower rate would cancelled amazon p...  


In [29]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['sentiment_label'] = le.fit_transform(df['sentiment'])

print(df.head())

                                              review  rating sentiment  \
0  I registered on the website, tried to order a ...     1.0  negative   
1  Had multiple orders one turned up and driver h...     1.0  negative   
2  I informed these reprobates that I WOULD NOT B...     1.0  negative   
3  I have bought from Amazon before and no proble...     1.0  negative   
4  If I could give a lower rate I would! I cancel...     1.0  negative   

                                        cleaned_text  sentiment_label  
0  registered website tried order laptop entered ...                0  
1  multiple order one turned driver phone door nu...                0  
2  informed reprobate would going visit sick rela...                0  
3  bought amazon problem happy service price amaz...                0  
4  could give lower rate would cancelled amazon p...                0  


In [30]:
X = df['cleaned_text']
y = df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [31]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [32]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [33]:
def evaluate_model(model_name, y_test, y_pred):
    print("\n", model_name)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

In [34]:
# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train_bow, y_train)
y_pred_lr = lr.predict(X_test_bow)
evaluate_model("Logistic Regression (BoW)", y_test, y_pred_lr)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)
evaluate_model("Naive Bayes (BoW)", y_test, y_pred_nb)

# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)
y_pred_dt = dt.predict(X_test_bow)
evaluate_model("Decision Tree (BoW)", y_test, y_pred_dt)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



 Logistic Regression (BoW)
Accuracy: 0.888387556399905
Precision: 0.8763860280747776
Recall: 0.888387556399905
F1 Score: 0.8809204904762741

 Naive Bayes (BoW)
Accuracy: 0.8684398005224412
Precision: 0.8694324132891854
Recall: 0.8684398005224412
F1 Score: 0.8676730047826531

 Decision Tree (BoW)
Accuracy: 0.781762051769176
Precision: 0.7837949398611899
Recall: 0.781762051769176
F1 Score: 0.7818146828635706


In [35]:
# Logistic Regression
lr.fit(X_train_tfidf, y_train)
y_pred_lr_tfidf = lr.predict(X_test_tfidf)
evaluate_model("Logistic Regression (TF-IDF)", y_test, y_pred_lr_tfidf)

# Naive Bayes
nb.fit(X_train_tfidf, y_train)
y_pred_nb_tfidf = nb.predict(X_test_tfidf)
evaluate_model("Naive Bayes (TF-IDF)", y_test, y_pred_nb_tfidf)

# Decision Tree
dt.fit(X_train_tfidf, y_train)
y_pred_dt_tfidf = dt.predict(X_test_tfidf)
evaluate_model("Decision Tree (TF-IDF)", y_test, y_pred_dt_tfidf)


 Logistic Regression (TF-IDF)
Accuracy: 0.9083353122773687
Precision: 0.911591507593799
Recall: 0.9083353122773687
F1 Score: 0.8905625280043827

 Naive Bayes (TF-IDF)
Accuracy: 0.8905248159582047
Precision: 0.8549491850689265
Recall: 0.8905248159582047
F1 Score: 0.8708035886173215


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



 Decision Tree (TF-IDF)
Accuracy: 0.8102588458798385
Precision: 0.807787695128815
Recall: 0.8102588458798385
F1 Score: 0.8086117700142268


In [36]:
results = pd.DataFrame({
    "Model": [
        "LR BoW", "NB BoW", "DT BoW",
        "LR TFIDF", "NB TFIDF", "DT TFIDF"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_lr_tfidf),
        accuracy_score(y_test, y_pred_nb_tfidf),
        accuracy_score(y_test, y_pred_dt_tfidf),
    ]
})

print(results.sort_values(by="Accuracy", ascending=False))

      Model  Accuracy
3  LR TFIDF  0.908335
4  NB TFIDF  0.890525
0    LR BoW  0.888388
1    NB BoW  0.868440
5  DT TFIDF  0.810259
2    DT BoW  0.781762


#Final Conclusion
1) Text preprocessing improved model performance.
2) TF-IDF performed better than Bag of Words.
3) Logistic Regression gave highest accuracy.
4) Naive Bayes was fastest.
5) Decision Tree showed overfitting.
6) Best combination: TF-IDF + Logistic Regression.